# Extract DOVER Features for SnapUGC 5K (Kaggle)\n\nExtract **technical score**, **aesthetic score**, and **pooled backbone features** from DOVER for all 5000 videos.\n\n**Output**: `dover_features.npz` with keys:\n- `ids`: video IDs\n- `technical_score`: (N,)\n- `aesthetic_score`: (N,)\n- `technical_feature`: (N, 768)\n- `aesthetic_feature`: (N, 768)

In [ ]:
# 1. Install dependencies\n!pip -q install timm einops thop opencv-python av --quiet\n!git clone https://github.com/VQAssessment/DOVER.git /tmp/DOVER --quiet

In [ ]:
import sys, os, numpy as np, pandas as pd, torch, cv2, warnings\nfrom pathlib import Path\nwarnings.filterwarnings('ignore')\nsys.path.insert(0, '/tmp/DOVER')\nfrom dover.models import DOVER\nfrom huggingface_hub import hf_hub_download\nfrom tqdm.auto import tqdm\nimport torch.nn.functional as F\n\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nprint('device:', device)

In [ ]:
# 2. Download DOVER weights\nckpt_path = hf_hub_download(repo_id='teowu/DOVER', filename='DOVER.pth', local_dir='/tmp/dover_weights')\nprint('weights:', ckpt_path)

In [ ]:
# 3. Load DOVER model\nmodel_args = {\n    'backbone': {\n        'technical': {'type': 'swin_tiny_grpb', 'checkpoint': True},\n        'aesthetic': {'type': 'conv_tiny'}\n    },\n    'backbone_preserve_keys': 'technical,aesthetic',\n    'divide_head': True,\n    'vqa_head': {'in_channels': 768, 'hidden_channels': 64}\n}\ndover = DOVER(**model_args).to(device).eval()\nstate = torch.load(ckpt_path, map_location=device)\ndover.load_state_dict(state, strict=True)\nprint('DOVER loaded')

In [ ]:
# 4. Setup paths\nVIDEO_DIR = '/kaggle/input/snapugc-videos/videos'\nCSV_PATH  = '/kaggle/input/snapugc-videos/train_subset_balanced_5000.csv'\nOUT_PATH  = '/kaggle/working/dover_features.npz'\n\ndf = pd.read_csv(CSV_PATH)\nvideo_ids = df['Id'].astype(str).tolist()\nprint('videos:', len(video_ids))

In [ ]:
# 5. Video loading with PyAV (cross-platform)\nimport av\n\ndef load_video_av(video_path, max_frames=None):\n    container = av.open(str(video_path))\n    frames = []\n    for i, frame in enumerate(container.decode(video=0)):\n        if max_frames and i >= max_frames:\n            break\n        arr = frame.to_ndarray(format='rgb24')\n        frames.append(torch.from_numpy(arr))\n    container.close()\n    if len(frames) == 0:\n        raise RuntimeError(f'No frames: {video_path}')\n    video = torch.stack(frames)\n    video = video.permute(3, 0, 1, 2).float()\n    return video\n\ndef get_spatial_fragments(video, fragments_h=7, fragments_w=7, fsize_h=32, fsize_w=32, aligned=32, random=False):\n    size_h = fragments_h * fsize_h\n    size_w = fragments_w * fsize_w\n    dur_t, res_h, res_w = video.shape[-3:]\n    ratio = min(res_h / size_h, res_w / size_w)\n    if ratio < 1:\n        ovideo = video\n        video = F.interpolate(video / 255.0, scale_factor=1 / ratio, mode='bilinear')\n        video = (video * 255.0).type_as(ovideo)\n    assert dur_t % aligned == 0, f'dur_t={dur_t} not divisible by aligned={aligned}'\n    hgrids = torch.LongTensor([min(res_h // fragments_h * i, res_h - fsize_h) for i in range(fragments_h)])\n    wgrids = torch.LongTensor([min(res_w // fragments_w * i, res_w - fsize_w) for i in range(fragments_w)])\n    hlength = res_h // fragments_h\n    wlength = res_w // fragments_w\n    if hlength > fsize_h:\n        rnd_h = torch.randint(hlength - fsize_h, (len(hgrids), len(wgrids), dur_t // aligned))\n    else:\n        rnd_h = torch.zeros((len(hgrids), len(wgrids), dur_t // aligned)).int()\n    if wlength > fsize_w:\n        rnd_w = torch.randint(wlength - fsize_w, (len(hgrids), len(wgrids), dur_t // aligned))\n    else:\n        rnd_w = torch.zeros((len(hgrids), len(wgrids), dur_t // aligned)).int()\n    target_video = torch.zeros(video.shape[:-2] + (size_h, size_w)).to(video.device)\n    for i, hs in enumerate(hgrids):\n        for j, ws in enumerate(wgrids):\n            for t in range(dur_t // aligned):\n                t_s, t_e = t * aligned, (t + 1) * aligned\n                h_s, h_e = i * fsize_h, (i + 1) * fsize_h\n                w_s, w_e = j * fsize_w, (j + 1) * fsize_w\n                h_so, h_eo = hs + rnd_h[i][j][t], hs + rnd_h[i][j][t] + fsize_h\n                w_so, w_eo = ws + rnd_w[i][j][t], ws + rnd_w[i][j][t] + fsize_w\n                target_video[:, t_s:t_e, h_s:h_e, w_s:w_e] = video[:, t_s:t_e, h_so:h_eo, w_so:w_eo]\n    return target_video\n\ndef temporal_sampling(total_frames, clip_len, num_clips, frame_interval):\n    all_inds = []\n    for clip_idx in range(num_clips):\n        start = int((total_frames - clip_len * frame_interval) * clip_idx / max(num_clips - 1, 1))\n        inds = np.arange(clip_len) * frame_interval + start\n        all_inds.append(inds)\n    return np.concatenate(all_inds).astype(np.int32)\n\ndef preprocess_dover_views(video_tensor, device='cpu'):\n    C, T, H, W = video_tensor.shape\n    mean = torch.FloatTensor([123.675, 116.28, 103.53]).to(device)\n    std = torch.FloatTensor([58.395, 57.12, 57.375]).to(device)\n    aest_frames = temporal_sampling(T, 32, 1, 2)\n    aest_frames = np.clip(aest_frames, 0, T - 1)\n    aest_video = video_tensor[:, aest_frames, :, :].to(device)\n    aest_video = F.interpolate(aest_video / 255.0, size=(224, 224), mode='bilinear')\n    aest_video = ((aest_video * 255.0) - mean.view(-1, 1, 1, 1)) / std.view(-1, 1, 1, 1)\n    aest_video = aest_video.unsqueeze(0)\n    tech_frames = temporal_sampling(T, 32, 3, 2)\n    tech_frames = np.clip(tech_frames, 0, T - 1)\n    tech_video = video_tensor[:, tech_frames, :, :].to(device)\n    if tech_video.shape[1] % 32 != 0:\n        pad = 32 - (tech_video.shape[1] % 32)\n        tech_video = F.pad(tech_video, (0, 0, 0, 0, 0, pad), mode='replicate')\n    tech_video = get_spatial_fragments(tech_video, fragments_h=7, fragments_w=7, fsize_h=32, fsize_w=32, aligned=32)\n    tech_video = ((tech_video / 255.0) - mean.view(-1, 1, 1, 1)) / std.view(-1, 1, 1, 1)\n    tech_video = tech_video.unsqueeze(0)\n    return {'aesthetic': aest_video, 'technical': tech_video}

In [ ]:
# 6. Extract loop with checkpointing\nresults = {\n    'ids': [],\n    'technical_score': [],\n    'aesthetic_score': [],\n    'technical_feature': [],\n    'aesthetic_feature': [],\n}\n\nfor i, vid in enumerate(tqdm(video_ids)):\n    vpath = Path(VIDEO_DIR) / f'{vid}.mp4'\n    if not vpath.exists():\n        vpath = Path(VIDEO_DIR) / vid\n    try:\n        video = load_video_av(vpath)\n        views = preprocess_dover_views(video, device=device)\n        with torch.no_grad():\n            scores, feats = dover(views, inference=True, return_pooled_feats=True)\n        tech_score = scores[0].mean().item()\n        aest_score = scores[1].mean().item()\n        tech_feat = feats['technical'].mean((-3, -2, -1)).cpu().numpy()\n        aest_feat = feats['aesthetic'].mean((-3, -2, -1)).cpu().numpy()\n        results['ids'].append(vid)\n        results['technical_score'].append(tech_score)\n        results['aesthetic_score'].append(aest_score)\n        results['technical_feature'].append(tech_feat)\n        results['aesthetic_feature'].append(aest_feat)\n    except Exception as e:\n        print(f'ERROR {vid}: {e}')\n        results['ids'].append(vid)\n        results['technical_score'].append(0.0)\n        results['aesthetic_score'].append(0.0)\n        results['technical_feature'].append(np.zeros(768, dtype=np.float32))\n        results['aesthetic_feature'].append(np.zeros(768, dtype=np.float32))\n    if (i + 1) % 200 == 0:\n        np.savez(OUT_PATH,\n                 ids=np.array(results['ids']),\n                 technical_score=np.array(results['technical_score'], dtype=np.float32),\n                 aesthetic_score=np.array(results['aesthetic_score'], dtype=np.float32),\n                 technical_feature=np.stack(results['technical_feature']).astype(np.float32),\n                 aesthetic_feature=np.stack(results['aesthetic_feature']).astype(np.float32))\n        print(f'Checkpoint saved at {i+1}/{len(video_ids)}')\n\nnp.savez(OUT_PATH,\n         ids=np.array(results['ids']),\n         technical_score=np.array(results['technical_score'], dtype=np.float32),\n         aesthetic_score=np.array(results['aesthetic_score'], dtype=np.float32),\n         technical_feature=np.stack(results['technical_feature']).astype(np.float32),\n         aesthetic_feature=np.stack(results['aesthetic_feature']).astype(np.float32))\nprint(f'Saved to {OUT_PATH}')